<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/11-capstone/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 11: Exercises

Twelve exercises: four Easy, five Medium and three Hard. Together they're the building blocks of the capstone: cleaning raw field data, validating it, preprocessing inside a pipeline, comparing and tuning models honestly, deciding when a model should say "unsure", checking for drift, and serving predictions through an API.

**Run the setup cell first.** It installs FastAPI and httpx if they're missing (Colab has everything else).

Rules:

- pandas, NumPy, scikit-learn, Pydantic and FastAPI.
- Solutions are in `03_solutions.ipynb`.

In [ ]:
# Setup: run this cell once before you start.
# It installs FastAPI and httpx if they're missing, and defines the check functions used below.

import importlib.util

_missing = [pkg for pkg in ["fastapi", "httpx"] if importlib.util.find_spec(pkg) is None]
if _missing:
    _packages = " ".join(_missing)
    %pip install -q {_packages}

import math

import numpy as np
import pandas as pd
from fastapi.testclient import TestClient
from pydantic import ValidationError
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _as_list(series):
    """Series values as a plain list, with every kind of missing value shown as None."""
    return [None if pd.isna(v) else v for v in series]


def check_snake_case(f):
    _scenarios("snake_case", [
        ("spaces and units", lambda: f("Culmen Length (mm)"), "culmen_length_mm", None),
        ("digits and slashes", lambda: f("Delta 15 N (o/oo)"), "delta_15_n_o_oo", "replace each run of other characters with one _"),
        ("camelCase", lambda: f("studyName"), "study_name", "insert _ between a lowercase letter or digit and a capital first"),
        ("a run of capitals stays together", lambda: f("Individual ID"), "individual_id", None),
        ("surrounding spaces", lambda: f("  Body Mass (g) "), "body_mass_g", "strip leading and trailing _"),
        ("already snake_case", lambda: f("body_mass_g"), "body_mass_g", None),
        ("a digit before a capital", lambda: f("delta15N"), "delta15_n", None),
    ])


def check_clean_species(f):
    raw = pd.Series(["Adelie Penguin (Pygoscelis adeliae)", "Gentoo penguin (Pygoscelis papua)",
                     "Chinstrap penguin (Pygoscelis antarctica)", "  adelie penguin "], index=[10, 11, 12, 13])
    _scenarios("clean_species", [
        ("full names to short names", lambda: _as_list(f(raw)), ["Adelie", "Gentoo", "Chinstrap", "Adelie"],
         "strip, split, take the first word, capitalise"),
        ("keeps the index", lambda: f(raw).index.tolist(), [10, 11, 12, 13], None),
        ("an unknown species raises", lambda: f(pd.Series(["Emperor penguin (Aptenodytes forsteri)"])), Raises(ValueError),
         "check the cleaned values are all in the list of three species"),
        ("a missing value raises", lambda: f(pd.Series(["Adelie Penguin", None])), Raises(ValueError), None),
    ])


def check_clean_sex(f):
    raw = pd.Series(["MALE", "FEMALE", " male ", "Female", ".", "", "unknown", None, np.nan], index=range(5, 14))
    _scenarios("clean_sex", [
        ("cleans valid values; everything else is missing", lambda: _as_list(f(raw)),
         ["male", "female", "male", "female", None, None, None, None, None],
         "strip and lowercase, then keep only 'male' and 'female' (Series.where)"),
        ("keeps the index", lambda: f(raw).index.tolist(), list(range(5, 14)), None),
        ("doesn't turn missing values into the text 'nan'", lambda: "nan" in [str(v).lower() for v in f(raw).dropna()], False,
         "astype(str) turns NaN into 'nan'; use the .str methods directly"),
    ])


def check_out_of_range(f):
    df = pd.DataFrame({"mass": [3700.0, 37.0, np.nan, 9000.0], "depth": [18.0, 175.0, 17.0, 5.0]}, index=["a", "b", "c", "d"])
    limits = {"mass": (2000, 7000), "depth": (10, 25)}
    expected = pd.DataFrame({"row": ["b", "b", "d", "d"], "column": ["mass", "depth", "mass", "depth"],
                             "value": [37.0, 175.0, 9000.0, 5.0]})
    _scenarios("out_of_range", [
        ("finds every value outside its limits, in order", lambda: f(df, limits), expected,
         "one line per bad value, sorted by row then by the column's position in limits"),
        ("limits are inclusive", lambda: len(f(pd.DataFrame({"x": [1.0, 2.0, 3.0]}), {"x": (1, 3)})), 0, None),
        ("missing values are skipped", lambda: len(f(pd.DataFrame({"x": [np.nan]}), {"x": (0, 1)})), 0, None),
        ("nothing wrong: empty with the right columns", lambda: list(f(df.loc[["a"]], limits).columns),
         ["row", "column", "value"], "return pd.DataFrame(columns=['row', 'column', 'value'])"),
    ])


def check_make_preprocessor(f):
    train = pd.DataFrame({"mass": [3000.0, 3500.0, np.nan, 5500.0], "depth": [15.0, 16.0, 17.0, 18.0],
                          "island": ["Biscoe", "Dream", np.nan, "Biscoe"], "notes": ["a", "b", "c", "d"]})
    new = pd.DataFrame({"mass": [4000.0], "depth": [16.5], "island": ["Torgersen"], "notes": ["x"]})

    filled = np.array([3000.0, 3500.0, 3500.0, 5500.0])         # the median of the known values is 3500
    median_filled = np.round((filled - filled.mean()) / filled.std(), 6).tolist()

    def unfitted():
        pre = f(["mass", "depth"], ["island"])
        return isinstance(pre, ColumnTransformer) and not hasattr(pre, "transformers_")

    def output():
        Z = f(["mass", "depth"], ["island"]).fit_transform(train)
        return (type(Z).__name__, Z.shape, bool(np.isnan(Z).any()))

    def scaled():
        Z = f(["mass", "depth"], ["island"]).fit_transform(train)
        return (np.round(Z[:, 0], 6).tolist(), bool(np.allclose(Z[:, :2].mean(axis=0), 0)))

    def onehot():
        Z = f(["mass", "depth"], ["island"]).fit_transform(train)
        return Z[:, 2:].tolist()

    def unknown():
        pre = f(["mass", "depth"], ["island"]).fit(train)
        return pre.transform(new)[0, 2:].tolist()

    _scenarios("make_preprocessor", [
        ("returns an unfitted ColumnTransformer", unfitted, True, "build it and return it; don't call fit"),
        ("dense array, extra columns dropped, no missing values", output, ("ndarray", (4, 4), False),
         "OneHotEncoder(sparse_output=False); imputers before the scaler and encoder"),
        ("numeric: median fill, then standardised", scaled, (median_filled, True),
         "SimpleImputer(strategy='median') then StandardScaler()"),
        ("categorical: most frequent fill, then one-hot", onehot,
         [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 0.0]], "SimpleImputer(strategy='most_frequent') before the encoder"),
        ("unseen category becomes all zeros", unknown, [0.0, 0.0], "OneHotEncoder(handle_unknown='ignore')"),
    ])


def check_compare_models(f):
    X, y = load_breast_cancer(return_X_y=True)
    X, y = X[:200], y[:200]
    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)
    models = {"baseline": DummyClassifier(),
              "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
              "knn": make_pipeline(StandardScaler(), KNeighborsClassifier())}
    expected_scores = {name: cross_val_score(m, X, y, cv=cv) for name, m in models.items()}

    def table():
        result = f(models, X, y, cv)
        return (list(result.columns), list(result.index)[-1], bool(result["mean"].is_monotonic_decreasing))

    def values():
        result = f(models, X, y, cv)
        return [(round(result.loc[n, "mean"], 6), round(result.loc[n, "std"], 6)) for n in models]

    _scenarios("compare_models", [
        ("columns, the baseline last, sorted best first", table, (["mean", "std"], "baseline", True),
         "sort_values('mean', ascending=False)"),
        ("means and standard deviations of the fold scores", values,
         [(round(s.mean(), 6), round(s.std(), 6)) for s in expected_scores.values()],
         "cross_val_score(model, X, y, cv=cv) for each model, with the cv passed in"),
        ("models are left unfitted", lambda: (f(models, X, y, cv), hasattr(models["knn"][-1], "classes_"))[1], False,
         "cross_val_score fits clones, not the originals"),
    ])


def check_selective_accuracy(f):
    y = np.array([1, 0, 1, 0, 1, 0])
    p = np.array([0.95, 0.10, 0.55, 0.52, 0.25, 0.80])
    _scenarios("selective_accuracy", [
        ("margin 0: always answers", lambda: f(y, p, 0.0), {"coverage": 1.0, "accuracy": 3 / 6, "n_covered": 6},
         "predicted class is 1 when p >= 0.5"),
        ("margin 0.2: abstains near 0.5", lambda: f(y, p, 0.2), {"coverage": 4 / 6, "accuracy": 2 / 4, "n_covered": 4},
         "covered when abs(p - 0.5) >= margin"),
        ("margin 0.4", lambda: f(y, p, 0.4), {"coverage": 2 / 6, "accuracy": 1.0, "n_covered": 2}, None),
        ("nothing covered: accuracy is nan", lambda: math.isnan(f(y, p, 0.49)["accuracy"]) and f(y, p, 0.49)["n_covered"] == 0,
         True, "return nan rather than dividing by zero"),
        ("n_covered is an int", lambda: type(f(y, p, 0.2)["n_covered"]), int, "int(covered.sum())"),
    ])


def check_PenguinIn(f):
    good = {"species": "Gentoo", "bill_length_mm": 46.5, "bill_depth_mm": 14.8, "flipper_length_mm": 217, "body_mass_g": 5200}

    def make(**changes):
        values = {**good, **changes}
        values = {k: v for k, v in values.items() if v is not ...}
        return f(**values)

    _scenarios("PenguinIn", [
        ("a valid penguin", lambda: make().model_dump(), {**good, "flipper_length_mm": 217.0, "body_mass_g": 5200.0}, None),
        ("numbers sent as text are converted", lambda: make(bill_length_mm="46.5").bill_length_mm, 46.5, None),
        ("limits are inclusive", lambda: make(body_mass_g=7000).body_mass_g, 7000.0, "Field(ge=2000, le=7000)"),
        ("unknown species is rejected", lambda: make(species="Emperor"), Raises(ValidationError),
         "species: Literal['Adelie', 'Chinstrap', 'Gentoo']"),
        ("body mass out of range is rejected", lambda: make(body_mass_g=37), Raises(ValidationError), None),
        ("bill depth out of range is rejected", lambda: make(bill_depth_mm=175), Raises(ValidationError), None),
        ("a missing field is rejected", lambda: make(flipper_length_mm=...), Raises(ValidationError), None),
        ("an unknown field is rejected", lambda: make(body_mass=5200), Raises(ValidationError),
         "model_config = ConfigDict(extra='forbid')"),
    ])


def check_drift_report(f):
    train = pd.DataFrame({"a": [1.0, 2.0, 3.0, 4.0, 5.0], "b": [10.0, 11.0, 12.0, np.nan, 13.0]})
    same = pd.DataFrame({"a": [2.0, 3.0, 4.0], "b": [11.0, 12.0, 11.5]})
    shifted = pd.DataFrame({"a": [5.0, 6.0, 7.0, np.nan], "b": [10.0, 11.0, 12.0, 13.0]})

    def report(new):
        r = f(train, new, ["a", "b"])
        return {col: (round(float(r.loc[col, "shift_sd"]), 4), round(float(r.loc[col, "outside_range"]), 4), bool(r.loc[col, "flag"]))
                for col in ["a", "b"]}

    sd_a = train["a"].std()
    _scenarios("drift_report", [
        ("no drift: nothing flagged", lambda: report(same),
         {"a": (0.0, 0.0, False), "b": (round((11.5 - 11.5) / train["b"].std(), 4), 0.0, False)}, None),
        ("a shifted column is flagged; missing values ignored", lambda: report(shifted),
         {"a": (round((6.0 - 3.0) / sd_a, 4), round(2 / 3, 4), True), "b": (0.0, 0.0, False)},
         "(new mean - train mean) / train std, and the fraction of new values outside [train min, train max]"),
        ("indexed by column name", lambda: list(f(train, same, ["b", "a"]).index), ["b", "a"], None),
    ])


def check_nested_cv_score(f):
    X, y = load_breast_cancer(return_X_y=True, as_frame=True)
    X, y = X.iloc[:240, :8], y.iloc[:240]
    estimator = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    grid = {"logisticregression__C": [0.01, 1.0, 100.0]}
    outer = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)
    inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)

    reference_scores, reference_params = [], []
    for tr, te in outer.split(X, y):
        s = GridSearchCV(estimator, grid, cv=inner).fit(X.iloc[tr], y.iloc[tr])
        reference_scores.append(s.score(X.iloc[te], y.iloc[te]))
        reference_params.append(s.best_params_)

    result = {}

    def run():
        if "value" not in result:
            result["value"] = f(estimator, grid, X, y, outer, inner)
        return result["value"]

    _scenarios("nested_cv_score", [
        ("one score per outer fold, matching a reference", lambda: run()[0], np.array(reference_scores),
         "fit GridSearchCV on X.iloc[train_idx] only, then .score on the test indices"),
        ("best parameters from each fold", lambda: run()[1], reference_params, "collect search.best_params_ each time"),
        ("the estimator passed in stays unfitted", lambda: hasattr(estimator[-1], "coef_"), False,
         "GridSearchCV clones it; don't fit the original"),
    ])


def check_permutation_importance(f):
    class OnlyUsesA:
        def predict(self, X):
            return (X["a"].to_numpy() > 0).astype(int)

    rng_data = np.random.default_rng(0)
    X = pd.DataFrame({"a": rng_data.normal(size=200), "b": rng_data.normal(size=200), "c": rng_data.normal(size=200)})
    y = (X["a"] > 0).astype(int).to_numpy()

    def result():
        return f(OnlyUsesA(), X, y, np.random.default_rng(1), n_repeats=3)

    def unchanged():
        before = X.copy()
        f(OnlyUsesA(), X, y, np.random.default_rng(2), n_repeats=2)
        return X.equals(before)

    def repeats_used():
        calls = []

        class Counting(OnlyUsesA):
            def predict(self, X):
                calls.append(1)
                return super().predict(X)
        f(Counting(), X, y, np.random.default_rng(3), n_repeats=4)
        return len(calls)

    def reproducible():
        return f(OnlyUsesA(), X, y, np.random.default_rng(5)).equals(f(OnlyUsesA(), X, y, np.random.default_rng(5)))

    _scenarios("permutation_importance", [
        ("sorted, with the used feature first", lambda: list(result().index), ["a", "b", "c"], "sort_values(ascending=False)"),
        ("shuffling the used feature costs about half the accuracy", lambda: 0.35 < result()["a"] < 0.65, True,
         "baseline accuracy minus the mean accuracy with that column shuffled"),
        ("features the model ignores have exactly zero importance", lambda: (result()["b"], result()["c"]), (0.0, 0.0),
         "shuffle only one column at a time, on a copy"),
        ("X is not modified", unchanged, True, "shuffle a copy: shuffled = X.copy()"),
        ("one baseline call plus n_repeats per column", repeats_used, 1 + 4 * 3, None),
        ("same seed, same result", reproducible, True, "use the rng passed in"),
    ])


def check_create_app(f):
    class FakeModel:
        """P(male) rises with body mass. Classes in the unusual order, male first."""
        classes_ = np.array(["male", "female"])

        def predict_proba(self, df):
            p_male = 1 / (1 + np.exp(-(df["body_mass_g"].to_numpy() - 4200) / 300))
            return np.column_stack([p_male, 1 - p_male])

    def penguin(mass, species="Adelie"):
        return {"species": species, "bill_length_mm": 40.0, "bill_depth_mm": 18.0, "flipper_length_mm": 195.0, "body_mass_g": mass}

    def expected(mass, margin=0.0):
        p = 1 / (1 + math.exp(-(mass - 4200) / 300))
        sex = "unsure" if abs(p - 0.5) < margin else ("male" if p >= 0.5 else "female")
        return {"sex": sex, "p_male": round(p, 3)}

    def client(margin=0.0):
        return TestClient(f(FakeModel(), margin=margin))

    def single(mass, margin=0.0):
        r = client(margin).post("/predict", json=penguin(mass))
        return (r.status_code, r.json())

    def batch(masses, margin=0.0):
        r = client(margin).post("/predict/batch", json={"penguins": [penguin(m) for m in masses]})
        return (r.status_code, r.json() if r.status_code == 200 else None)

    _scenarios("create_app", [
        ("GET /health", lambda: client().get("/health").json(), {"status": "ok"}, None),
        ("a heavy penguin is male", lambda: single(5200), (200, expected(5200)),
         "find the male column with list(model.classes_).index('male')"),
        ("a light penguin is female", lambda: single(3300), (200, expected(3300)), None),
        ("near 0.5 with a margin: unsure", lambda: single(4250, margin=0.1), (200, expected(4250, 0.1)),
         "'unsure' when abs(p_male - 0.5) < margin"),
        ("invalid input is rejected with 422", lambda: client().post("/predict", json=penguin(37)).status_code, 422,
         "use PenguinIn as the argument type"),
        ("batch: one result per penguin, in order", lambda: batch([5200, 3300, 4250], margin=0.1),
         (200, {"predictions": [expected(5200, 0.1), expected(3300, 0.1), expected(4250, 0.1)]}), None),
        ("an empty batch is rejected", lambda: batch([])[0], 422, "Field(min_length=1, max_length=100)"),
        ("more than 100 penguins is rejected", lambda: batch([4000] * 101)[0], 422, None),
    ])


print("Setup done. Checks are ready.")

## Easy

### Exercise 1: `snake_case`

Raw data rarely has convenient column names. Write `snake_case(name)` that turns a column name into lowercase `snake_case`:

```
"Culmen Length (mm)"   ->  "culmen_length_mm"
"Delta 15 N (o/oo)"    ->  "delta_15_n_o_oo"
"studyName"            ->  "study_name"
"  Body Mass (g) "     ->  "body_mass_g"
```

- A lowercase letter or digit followed by an uppercase letter starts a new word (`studyName`), but a run of capitals like `ID` stays together.
- Every run of characters that aren't letters or digits becomes a single `_`.
- No `_` at the start or end.

In [ ]:
def snake_case(name: str) -> str:
    # your code here
    pass

In [ ]:
check_snake_case(snake_case)

### Exercise 2: `clean_species`

In the raw data, species are recorded as full names, like `"Adelie Penguin (Pygoscelis adeliae)"` or `"Gentoo penguin (Pygoscelis papua)"`.

Write `clean_species(series)` that returns a Series (same index) with just `"Adelie"`, `"Chinstrap"` or `"Gentoo"`:

- Use the first word, ignoring case and surrounding spaces, and return it capitalised.
- If any value is missing, or isn't one of the three species, raise `ValueError` listing the bad values. Don't guess.

In [ ]:
def clean_species(series: pd.Series) -> pd.Series:
    # your code here
    pass

In [ ]:
check_clean_species(clean_species)

### Exercise 3: `clean_sex`

The `Sex` column in raw field data may contain `"MALE"`, `"FEMALE"`, lowercase versions, extra spaces, and placeholders like `"."`, `""` or `"unknown"` for birds that couldn't be sexed.

Write `clean_sex(series)` that returns a Series (same index) where:

- male and female values, in any case and with any surrounding spaces, become `"male"` and `"female"`;
- everything else, including missing values, becomes missing (`NaN`).

In [ ]:
def clean_sex(series: pd.Series) -> pd.Series:
    # your code here
    pass

In [ ]:
check_clean_sex(clean_sex)

### Exercise 4: `out_of_range`

Write `out_of_range(df, limits)` that finds implausible values.

- `limits` maps column names to `(low, high)`, both inclusive. For example `{"body_mass_g": (2000, 7000)}`.
- Return a DataFrame with columns `"row"` (the index label), `"column"` and `"value"`, one line per value outside its limits, sorted by row and then by the order of the columns in `limits`. Reset the index.
- Missing values are not "out of range" (they're a different problem): skip them.
- If nothing is out of range, return an empty DataFrame with those three columns.

In [ ]:
def out_of_range(df: pd.DataFrame, limits: dict) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_out_of_range(out_of_range)

## Medium

### Exercise 5: `make_preprocessor`

Write `make_preprocessor(numeric, categorical)` that returns an **unfitted** scikit-learn `ColumnTransformer` that:

- for the `numeric` columns: fills missing values with the column **median**, then standardises;
- for the `categorical` columns: fills missing values with the **most frequent** value, then one-hot encodes, with categories never seen in training encoded as all zeros instead of raising an error;
- returns a **dense** NumPy array (set `sparse_output=False` on the encoder);
- drops any other columns.

Numeric columns come first in the output, then the one-hot columns.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_preprocessor(numeric: list, categorical: list) -> ColumnTransformer:
    # your code here
    pass

In [ ]:
check_make_preprocessor(make_preprocessor)

### Exercise 6: `compare_models`

Write `compare_models(models, X, y, cv)` that cross-validates several models and returns a summary.

- `models` is a dict mapping a name to an unfitted estimator (or pipeline).
- Use `cross_val_score` with the given `cv` (accuracy, the default score).
- Return a DataFrame indexed by model name, with columns `"mean"` and `"std"` of the fold scores, sorted by `"mean"`, best first.

In [ ]:
from sklearn.model_selection import cross_val_score


def compare_models(models: dict, X, y, cv) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_compare_models(compare_models)

### Exercise 7: `selective_accuracy`

A field tool can answer "unsure" instead of guessing. Write `selective_accuracy(y_true, p, margin)` that measures what that buys.

- `y_true` holds 0/1 labels and `p` the predicted probabilities of class 1.
- A prediction is **made** when `|p - 0.5| >= margin`; otherwise the model abstains. A made prediction is class 1 if `p >= 0.5`.
- Return a dict with `"coverage"` (the fraction of predictions made), `"accuracy"` (accuracy on the predictions made, or `nan` if none were made) and `"n_covered"` (how many were made, as an `int`).

`margin=0` means the model always answers.

In [ ]:
def selective_accuracy(y_true, p, margin: float) -> dict:
    # your code here
    pass

In [ ]:
check_selective_accuracy(selective_accuracy)

### Exercise 8: `PenguinIn`

Write a Pydantic model `PenguinIn` describing one penguin sent to the capstone API:

- `species`: exactly one of `"Adelie"`, `"Chinstrap"`, `"Gentoo"` (use `typing.Literal`).
- `bill_length_mm` between 25 and 70, `bill_depth_mm` between 10 and 25, `flipper_length_mm` between 160 and 240, `body_mass_g` between 2000 and 7000, all floats, all limits inclusive (`Field(ge=..., le=...)`).
- **Unknown fields are an error** (`model_config = ConfigDict(extra="forbid")`), so a misspelled field name isn't silently ignored.

Invalid input should raise `pydantic.ValidationError`.

In [ ]:
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field


class PenguinIn(BaseModel):
    # your code here
    pass

In [ ]:
check_PenguinIn(PenguinIn)

### Exercise 9: `drift_report`

Write `drift_report(train, new, columns)` that compares new data with the training data, column by column. Return a DataFrame indexed by column name with:

- `"shift_sd"`: `(mean of new - mean of train) / standard deviation of train`, using pandas' default `std()`;
- `"outside_range"`: the fraction of non-missing new values below the training minimum or above the training maximum;
- `"flag"`: `True` if `|shift_sd| > 0.5` or `outside_range > 0.05`.

Ignore missing values everywhere.

In [ ]:
def drift_report(train: pd.DataFrame, new: pd.DataFrame, columns: list) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_drift_report(drift_report)

## Hard

### Exercise 10: `nested_cv_score`

Write `nested_cv_score(estimator, param_grid, X, y, outer_cv, inner_cv)` that gives an honest estimate for "this estimator, with hyperparameters chosen by grid search".

For each split of `outer_cv`:

1. Run `GridSearchCV(estimator, param_grid, cv=inner_cv)` on the **training** part only.
2. Score the search's best model on the **test** part (`.score`, accuracy for classifiers).

Return `(scores, best_params)`: a NumPy array with one score per outer fold, and a list with the best parameters chosen in each fold.

`X` is a DataFrame and `y` a Series: use `.iloc` with the split's indices.

In [ ]:
from sklearn.model_selection import GridSearchCV


def nested_cv_score(estimator, param_grid: dict, X, y, outer_cv, inner_cv):
    # your code here
    pass

In [ ]:
check_nested_cv_score(nested_cv_score)

### Exercise 11: `permutation_importance`

Which features does a fitted model actually rely on? **Permutation importance** answers by breaking one feature at a time: shuffle a column, and see how much accuracy drops. A feature the model ignores gives no drop at all.

Write `permutation_importance(model, X, y, rng, n_repeats=5)`:

- `model` is fitted; `X` is a DataFrame; `y` the true labels; `rng` a NumPy `Generator`.
- Compute the baseline accuracy (`model.predict` on `X`).
- For each column: `n_repeats` times, shuffle **that column only** (on a copy of `X`), compute the accuracy, and record `baseline - accuracy`.
- Return a Series indexed by column name with the **mean** drop, sorted from largest to smallest.
- Don't modify `X`.

In [ ]:
def permutation_importance(model, X: pd.DataFrame, y, rng, n_repeats: int = 5) -> pd.Series:
    # your code here
    pass

In [ ]:
check_permutation_importance(permutation_importance)

### Exercise 12: `create_app`

Write `create_app(model, margin=0.0)` that returns a FastAPI app serving a fitted sex classifier. `model` has `predict_proba(df)` and `classes_` (containing `"female"` and `"male"`, **in either order**).

Endpoints:

- `GET /health` returns `{"status": "ok"}`.
- `POST /predict` takes one penguin (use `PenguinIn` from exercise 8) and returns `{"sex": ..., "p_male": ...}`. `p_male` is the probability of `"male"` rounded to 3 decimals. `sex` is `"male"` if `p_male >= 0.5`, `"female"` otherwise, or `"unsure"` if `|p_male - 0.5| < margin`.
- `POST /predict/batch` takes `{"penguins": [...]}`, a list of 1 to 100 penguins, and returns `{"predictions": [...]}`, one result per penguin, in order. An empty list or more than 100 must be rejected with status 422 (use `Field(min_length=1, max_length=100)`).

Build the model's input with `pd.DataFrame([p.model_dump() for p in penguins])`. Find the `"male"` column of `predict_proba` from `model.classes_`; don't assume it's the second.

In [ ]:
from fastapi import FastAPI


def create_app(model, margin: float = 0.0) -> FastAPI:
    # your code here
    pass

In [ ]:
check_create_app(create_app)

## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then open the capstone in `04_capstone.ipynb`, where these pieces become a complete system: from the raw Palmer Station field records to a tested prediction API.